In [1]:
import os, pandas as pd, matplotlib.pyplot as plt
os.chdir(os.path.expanduser("~/growth-marketing-analytics"))
ga = pd.read_parquet("data/processed/ga_sessions.parquet")
ga["new_visitor"] = ga.visit_number == 1
OUT = "module_2_google_analytics/outputs/"
print(ga.shape)

TypeError: data type 'dbdate' not understood

In [2]:
import os, pandas as pd, matplotlib.pyplot as plt
import db_dtypes
os.chdir(os.path.expanduser("~/growth-marketing-analytics"))
ga = pd.read_parquet("data/processed/ga_sessions.parquet")

ga["date"] = pd.to_datetime(ga["date"].astype(str))
ga.to_parquet("data/processed/ga_sessions.parquet", index=False)

ga["new_visitor"] = ga.visit_number == 1
OUT = "module_2_google_analytics/outputs/"
print(ga.shape, ga.date.dtype)

(903653, 19) datetime64[us]


In [3]:
buy = ga.transactions > 0
kpi = pd.Series({
    "Sessions": len(ga),
    "Users": ga.visitor_id.nunique(),
    "Bounce rate %": ga.bounce.mean()*100,
    "Pages per session": ga.pageviews.mean(),
    "Conversion rate %": buy.mean()*100,
    "Orders": ga.transactions.sum(),
    "Revenue $": ga.revenue.sum(),
    "AOV $": ga.revenue.sum()/ga.transactions.sum(),
    "Revenue per session $": ga.revenue.sum()/len(ga),
}).round(2)
kpi.to_csv(OUT + "overall_kpis.csv")
kpi

Sessions                  903653.00
Users                     714167.00
Bounce rate %                 49.87
Pages per session              3.85
Conversion rate %              1.28
Orders                     12115.00
Revenue $                1540071.24
AOV $                        127.12
Revenue per session $          1.70
dtype: float64

In [4]:
ch = ga.groupby("channel").agg(
    sessions=("session_id","count"),
    users=("visitor_id","nunique"),
    bounce_rate=("bounce","mean"),
    pages_per_session=("pageviews","mean"),
    buying_sessions=("transactions", lambda x: (x>0).sum()),
    orders=("transactions","sum"),
    revenue=("revenue","sum"),
    new_visitor_share=("new_visitor","mean"),
)
ch["session_share_%"] = ch.sessions / ch.sessions.sum() * 100
ch["revenue_share_%"] = ch.revenue / ch.revenue.sum() * 100
ch["conversion_rate_%"] = ch.buying_sessions / ch.sessions * 100
ch["aov"] = ch.revenue / ch.orders
ch["revenue_per_session"] = ch.revenue / ch.sessions
ch["bounce_rate"] *= 100
ch["new_visitor_share"] *= 100
ch = ch.sort_values("revenue", ascending=False).round(2)
ch.to_csv(OUT + "channel_scorecard.csv")
ch[["sessions","session_share_%","revenue","revenue_share_%","conversion_rate_%","aov","revenue_per_session","bounce_rate"]]

KeyError: "Label(s) ['session_id'] do not exist"

In [ ]:
import os, pandas as pd, matplotlib.pyplot as plt
os.chdir(os.path.expanduser("~/growth-marketing-analytics"))
ga = pd.read_parquet("data/processed/ga_sessions.parquet")

ga["session_id"] = ga.visitor_id + "_" + ga.visit_id.astype(str)
ga.to_parquet("data/processed/ga_sessions.parquet", index=False)

ga["new_visitor"] = ga.visit_number == 1
OUT = "module_2_google_analytics/outputs/"
print(ga.shape, ga.date.dtype)
